In [1]:
from confluent_kafka import Producer
import pandas as pd
import json

In [2]:
data = pd.read_csv('./input/sorted_streaming_data.csv')
data['SensorID'] = data['SensorID'].astype(str)

In [3]:
# Kafka configuration
kafka_config = {
    'bootstrap.servers': 'localhost:9092',  # Adjust this to your Kafka server address
}

# Initialize the Kafka producer
producer = Producer(kafka_config)

In [4]:
# Define the Kafka topic
topic = 'water-quality'  # Replace with your actual Kafka topic


In [5]:

# Function to deliver messages
def delivery_report(err, msg):
    if err is not None:
        print(f"Message delivery failed: {err}")
    else:
        print(f"Message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")


In [6]:
import time 

# Function to send DataFrame records to Kafka
def send_records_to_kafka(df):
    
    for _, row in df.iterrows():
        time.sleep(1)
        
        record = {
            'device_id': row['SensorID'],
            'timestamp': row['DateTime'],
            'water_turbidity': row['Turbidity']
        }
        
        # Convert the record to JSON format
        record_json = json.dumps(record)

        # Send the record to Kafka
        producer.produce(
            topic, 
            key=row['SensorID'], 
            value=record_json, 
            callback=delivery_report
        )

        # Wait for any outstanding messages to be delivered and delivery reports to be received
        producer.flush()




In [10]:
# Send the records from the DataFrame to Kafka
send_records_to_kafka(data)

Message delivered to water-quality [0] at offset 100123
Message delivered to water-quality [0] at offset 100124
Message delivered to water-quality [0] at offset 100125


KeyboardInterrupt: 